# B2-023 — Practice p23

*55 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** challenge · **Difficulty:** advanced · **Time budget:** 55 minutes  
**Concepts:** denoising-diffusion-probabilistic-models

## Task

A teammate's DDPM implementation has three independent defects: an $\alpha$/$\bar\alpha$ confusion, a wrong sampling $\sigma_t$, and an off-by-one timestep index. It uses this unit's schedule convention (`make_schedule` with the p09 contract, $(T+1)$-length tensors with padding at index $0$) and the pinned sampling variance $\sigma_t^2=\tilde\beta_t$.

```python
def q_sample(x0, t, eps, schedule):
    a = schedule["alpha"][t].to(x0.dtype).unsqueeze(1)
    return a.sqrt() * x0 + (1 - a).sqrt() * eps

def p_sample(model, x_t, t, z, schedule):
    T = schedule["beta"].shape[0] - 1
    beta, alpha, alpha_bar = (schedule[k][t].item() for k in ("beta", "alpha", "alpha_bar"))
    eps_hat = model(x_t, torch.full((x_t.shape[0], 1), t / T, dtype=x_t.dtype))
    mean = (x_t - beta / math.sqrt(1 - alpha_bar) * eps_hat) / math.sqrt(alpha)
    if t == 1:
        return mean
    sigma = schedule["beta_tilde"][t].item()
    return mean + sigma * z

def sample_loop(model, x_T, noises, schedule):
    T = schedule["beta"].shape[0] - 1
    x = x_T
    for t in range(T, 1, -1):
        x = p_sample(model, x, t, noises[t - 1], schedule)
    return x
```

(A) Copy the code as given (rename the functions with a `buggy_` prefix) and run it on the literals in the supplied cell with `schedule = make_schedule(4, 0.1, 0.4)`: report `buggy_q_sample(X0, T_IDX, EPS, schedule)` and `buggy_sample_loop(ToyEps(), X_T, NOISES, schedule)`.
(B) Identify each defect precisely: quote the expression, name the defect class, and state what Sessions 3–4 require instead.
(C) Implement repaired `q_sample(x0, t, eps, schedule)`, `p_sample(model, x_t, t, z, schedule)`, and `sample_loop(model, x_T, noises, schedule)` with the **same signatures**. On the literals they must give (`ATOL = RTOL = 1e-5`): `q_sample(X0, T_IDX, EPS, schedule) = [[2.2262065411, -0.3193778992], [-0.4176122546, 1.2424757481]]` and `sample_loop(ToyEps(), X_T, NOISES, schedule) = [[3.5601835251, 1.2876268625], [0.7925575972, 1.8815526962]]`.
(D) For each defect separately, reintroduce only that defect into your repaired code and report the resulting `q_sample` or `sample_loop` output; assert that each single-defect variant differs from the repaired value by more than `0.05` (maximum absolute difference).
(E) Explain (i) why a probe at $t=1$ cannot detect the $\alpha$/$\bar\alpha$ defect; (ii) why using $\tilde\beta_t$ as the standard deviation is wrong, and how it differs from the legitimate alternative $\sigma_t^2=\beta_t$; and (iii) what the buggy loop returns instead of $x_0$ and why its output still "looks like" a sample.

**Required answer-check assertions.** Both repaired values, the three single-defect differences, a recording-model check that the repaired loop visits `t_scaled` values `1.0, 0.75, 0.5, 0.25` in that order, and `torch.equal` of the repaired $t=1$ step with and without its noise argument.

In [ ]:
import math
import torch
from torch import nn

X0 = torch.tensor([[2.0, -1.0], [0.0, 1.5]])
T_IDX = torch.tensor([2, 4])
EPS = torch.tensor([[1.0, 1.0], [-0.5, 0.5]])
X_T = torch.tensor([[1.5, -0.5], [-1.0, 1.0]])
NOISES = torch.tensor([[[0.3, -0.3], [0.1, 0.1]],
                       [[1.0, 0.5], [-0.5, 1.0]],
                       [[-0.2, 0.4], [0.6, -0.8]],
                       [[0.5, 0.5], [1.0, -1.0]]])


class ToyEps(nn.Module):
    def forward(self, x_t, t_scaled):
        return 0.25 * x_t - t_scaled


ATOL = 1e-5
RTOL = 1e-5

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.